# SST Indices & Native ELI Preprocessing

This workflow generates regional SST indices (14 standard and derived regions) and native MPAS-Ocean Equatorial Longitude Index (ELI) time series across **E3SM**, **CESM-SMYLE**, and **HadISST2** observations.

> **CLI Alternative**:
> This workflow can be run entirely in batch mode without Jupyter via:
> ```bash
> python scripts/run_process_sst_index.py --sources obs smyle e3sm --eli-grid both
> ```
> Or for native MPAS-Ocean ELI specifically:
> ```bash
> python scripts/run_process_native_eli.py --cases JRA55_FOSIRL Reanalysis
> ```

Preprocessed output files are written to `$S2D_DIAG_ROOT/{case}/sst_index/timeseries/`.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
import subprocess
from pathlib import Path
import numpy as np
import xarray as xr

# Resolve the checkout from the repository root or any nested notebook directory.
_working_directory = Path.cwd().resolve()
REPO_ROOT = next(
    (candidate for candidate in (_working_directory, *_working_directory.parents)
     if (candidate / "scripts" / "run_process_sst_index.py").is_file()
     and (candidate / "esp_lab").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not locate the ESP-Lab repository root.")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from esp_lab.paths import S2D_DIAG_ROOT, HADISST2_DIAG_DIR, CESM_SMYLE_DIAG_DIR
from esp_lab.diagnostics.sst_index import VALID_REGIONS
from esp_lab.diagnostics.native_eli import DEFAULT_MPAS_MESH_FILE


In [ ]:
# -----------------------------
# Pipeline Configuration
# -----------------------------
CONFIG = {
    # Data sources to process: "obs", "smyle", and/or "e3sm"
    "sources": ["obs", "smyle", "e3sm"],
    # SST index regions to compute (default: all valid regions)
    "regions": VALID_REGIONS,
    # Grid for ELI index: 'regridded', 'native', or 'both'
    "eli_input_grid": "both",
    "force": False,
    "init_months": [5, 11],
    "year_start": 1980,
    "year_end": 2018,
    "climy0": 1981,
    "climy1": 2010,
    "nlead": 24,
    "e3sm_nens": 10,
    "smyle_nens": 20,
    "workers": 8,
    "outdir": str(S2D_DIAG_ROOT),
    "smyle_outdir": str(CESM_SMYLE_DIAG_DIR),
    "obs_outdir": str(HADISST2_DIAG_DIR / "sst_index" / "timeseries"),
    "mesh_file": str(DEFAULT_MPAS_MESH_FILE),
    "e3sm_cases": {
        "JRA55_FOSIRL": {
            "display_name": "E3SM-FOSIRL",
            "data_dir": "/global/cfs/cdirs/e3sm/S2S2D/post_process",
            "eli_data_dir": "/global/cfs/cdirs/e3smdata/simulations/S2S2D",
            "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_JRA55_FOSIRL",
            "cache_tag": "JRA55_FOSIRL",
        },
        "Reanalysis": {
            "display_name": "E3SM-Reanalysis",
            "data_dir": "/global/cfs/cdirs/e3sm/S2S2D/post_process",
            "eli_data_dir": "/global/cfs/cdirs/e3sm/S2S2D/simulation",
            "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_Reanalysis",
            "cache_tag": "Reanalysis",
        },
        "4DEnVarOcn": {
            "display_name": "E3SM-4DEnVarOcn",
            "data_dir": "/global/cfs/cdirs/e3sm/S2S2D/post_process",
            "eli_data_dir": "/global/cfs/cdirs/e3sm/S2S2D/simulation",
            "case_prefix": "WCYCL20TR_ne30pg2_r05_IcoswISC30E3r5_4DEnVarOcn",
            "cache_tag": "4DEnVarOcn",
            "year_end": 2011,
        },
    },
    "auto_generate_smyle_benchmark": True,
}


In [ ]:
# -----------------------------
# Execute Preprocessing Pipeline
# -----------------------------
script_path = str(REPO_ROOT / "scripts" / "run_process_sst_index.py")

# Ensure GDAL/PROJ library paths
env = os.environ.copy()
conda_prefix = sys.prefix
env["GDAL_DATA"] = f"{conda_prefix}/share/gdal"
env["PROJ_LIB"] = f"{conda_prefix}/share/proj"

def build_base_cmd(sources, regions):
    cmd = [
        sys.executable,
        script_path,
        "--sources", *sources,
        "--outdir", CONFIG["outdir"],
        "--smyle-outdir", CONFIG["smyle_outdir"],
        "--obs-outdir", CONFIG["obs_outdir"],
        "--init-months", *(str(m) for m in CONFIG["init_months"]),
        "--year-start", str(CONFIG["year_start"]),
        "--year-end", str(CONFIG["year_end"]),
        "--climy0", str(CONFIG["climy0"]),
        "--climy1", str(CONFIG["climy1"]),
        "--nlead", str(CONFIG["nlead"]),
        "--e3sm-nens", str(CONFIG["e3sm_nens"]),
        "--smyle-nens", str(CONFIG["smyle_nens"]),
        "--workers", str(CONFIG["workers"]),
        "--regions", *regions,
        "--eli-grid", str(CONFIG.get("eli_input_grid", "both")),
        "--mesh-file", str(CONFIG.get("mesh_file", DEFAULT_MPAS_MESH_FILE)),
    ]
    if CONFIG["force"]:
        cmd.append("--force")
    return cmd

def run_cmd(cmd, label):
    print("=" * 60)
    print(f"Processing: {label}")
    print("=" * 60)
    print(" ".join(cmd))
    result = subprocess.run(cmd, env=env, text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Failed for '{label}' with exit code {result.returncode}")

# Process shared sources (obs, smyle)
shared_sources = [s for s in CONFIG["sources"] if s != "e3sm"]
if shared_sources:
    cmd = build_base_cmd(shared_sources, CONFIG["regions"])
    run_cmd(cmd, ", ".join(shared_sources))

# Process E3SM cases
if "e3sm" in CONFIG["sources"]:
    for case_key, case_info in CONFIG["e3sm_cases"].items():
        cmd = build_base_cmd(["e3sm"], CONFIG["regions"])
        cmd.extend([
            "--e3sm-data-dir", case_info["data_dir"],
            "--e3sm-case-prefix", case_info["case_prefix"],
            "--e3sm-cache-tag", case_info["cache_tag"],
            "--e3sm-display-name", case_info.get("display_name", case_key),
            "--e3sm-raw-dir", case_info.get("eli_data_dir", ""),
            "--year-end", str(case_info.get("year_end", CONFIG["year_end"])),
        ])
        run_cmd(cmd, f"E3SM case: {case_key}")

print("\nAll SST and ELI indices processed successfully!")


In [ ]:
# -----------------------------
# Verification & Summary Inspection
# -----------------------------
import matplotlib.pyplot as plt

diag_root = Path(CONFIG["outdir"])
print(f"Checking generated products in {diag_root}:\n")

for case_key, case_info in CONFIG["e3sm_cases"].items():
    timeseries_dir = diag_root / case_info["cache_tag"] / "sst_index" / "timeseries"
    print(f"=== {case_key}: {timeseries_dir} ===")
    files = sorted(timeseries_dir.glob("*.nc"))
    print(f"  Total product files: {len(files)}")
    for init_month in CONFIG["init_months"]:
        native_eli = timeseries_dir / f"E3SMLE{init_month:02d}_ELI_native_N{CONFIG['e3sm_nens']:02d}_M{CONFIG['nlead']:02d}.nc"
        regrid_eli = timeseries_dir / f"E3SMLE{init_month:02d}_ELI_N{CONFIG['e3sm_nens']:02d}_M{CONFIG['nlead']:02d}.nc"
        nino34 = timeseries_dir / f"E3SMLE{init_month:02d}_TS_N{CONFIG['e3sm_nens']:02d}_M{CONFIG['nlead']:02d}_Nino3.4SST_mon.nc"
        print(f"  Init {init_month:02d} Native ELI  : {'[OK]' if native_eli.exists() else '[MISSING]'}")
        print(f"  Init {init_month:02d} Regrid ELI  : {'[OK]' if regrid_eli.exists() else '[MISSING]'}")
        print(f"  Init {init_month:02d} Nino 3.4 SST: {'[OK]' if nino34.exists() else '[MISSING]'}")
